# 7-Eleven NPD Framework: Integrated EDA Master Notebook
본 노트북은 프로젝트 내 여러 폴더에 산재해 있던 핵심 EDA 스크립트와 주피터 노트북의 파편들을 하나의 통합된 흐름으로 정리한 문서입니다.


## 1. 신상품 식별 및 성공/실패 라벨링 (NPD Identification & Labeling)
- **핵심 로직:** POS(B2) 데이터에서 상품별 '최초 결제 발생일'을 추출하여 신상품군을 확정하고, 출시 후 4주 누적 매출을 기준으로 분석합니다.


In [1]:
import polars as pl
import datetime
import os
import pandas as pd

B2_PATH = r"../data/processed/POS 전처리 최종/pos_data_food_final_상품단위변환전.parquet"
B4_PATH = r"../data/processed/B4_ITEM_DV_INFO.parquet"

def get_true_npd_list(b2_path, b4_path, burn_in_days=14):
    print(f"--- [Step 1.0] 식품 전용 순수 NPD 식별 로직 ---")
    food_categories = ['음료', '과자', '유음료', '미반', '면', '냉장', '맥주', '즉석음료', '빵', '전통주', '아이스크림', '조리빵', '즉석 식품', '건강/기호식품', '가공식품', '양주와인', '디저트', '안주', '신선', '간식', '조미료/건물', '냉동']
    
    try: b4_df = pl.read_parquet(b4_path)
    except: b4_df = pl.read_csv(b4_path.replace('.parquet', '.csv'), ignore_errors=True)
    
    if "ITEM_CD" in b4_df.columns: b4_df = b4_df.rename({"ITEM_CD": "상품코드"})
    b4_df = b4_df.with_columns(pl.col("상품코드").cast(pl.Utf8))
    b4_food = b4_df.filter(pl.col("ITEM_LGDV_NM").is_in(food_categories))
    food_item_list = b4_food.select("상품코드").to_series().to_list()
    
    b2_lazy = pl.scan_parquet(b2_path).filter(pl.col("상품코드").is_in(food_item_list))
    start_date = b2_lazy.select(pl.col("영업일자").min()).collect().item()
    start_dt = datetime.datetime.strptime(str(start_date), "%Y%m%d")
    burn_in_threshold = int((start_dt + datetime.timedelta(days=burn_in_days)).strftime("%Y%m%d"))
    
    legacy_items = b2_lazy.filter(pl.col("영업일자") <= burn_in_threshold).select("상품코드").unique().collect()
    legacy_set = set(legacy_items["상품코드"].to_list())
    
    npd_launch_df = b2_lazy.filter(~pl.col("상품코드").is_in(list(legacy_set))).group_by("상품코드").agg(pl.col("영업일자").min().alias("launch_dt")).collect()
    npd_launch_df = npd_launch_df.with_columns([pl.col("launch_dt").cast(pl.Utf8).str.to_date("%Y%m%d").alias("launch_date")])
    npd_launch_df = npd_launch_df.with_columns([(pl.col("launch_date") + pl.duration(days=30)).alias("end_date")])
    npd_launch_df = npd_launch_df.with_columns([pl.col("end_date").dt.strftime("%Y%m%d").cast(pl.Int64).alias("end_dt")])
    
    b2_npd_lazy = b2_lazy.filter(pl.col("상품코드").is_in(npd_launch_df["상품코드"].to_list()))
    b2_joined = b2_npd_lazy.join(npd_launch_df.lazy(), on="상품코드", how="inner")
    
    npd_sales = b2_joined.filter((pl.col("영업일자") >= pl.col("launch_dt")) & (pl.col("영업일자") <= pl.col("end_dt"))).group_by("상품코드").agg(pl.col("매출금액").sum().alias("1month_sales")).collect()
    valid_npd_df = npd_sales.filter(pl.col("1month_sales") > 0)
    valid_npd_set = set(valid_npd_df["상품코드"].to_list())
    
    out_path = "00_TRUE_NPD_LIST.xlsx"
    b4_food.filter(pl.col("상품코드").is_in(list(valid_npd_set))).to_pandas().to_excel(out_path, index=False)
    return valid_npd_set, legacy_set, food_item_list, b4_food

TRUE_NPD_SET, LEGACY_SET, FOOD_ITEMS, B4_FOOD_LAZY = get_true_npd_list(B2_PATH, B4_PATH)

CLUSTER_COLOR_MAP = {0: '#FCE378', 1: '#FF8C42', 2: '#EF4444', 3: 'goldenrod'}

## 2. 신상품 1개월 초기 성과 기준 파레토 분석


In [3]:
print("신상품 1개월 매출 기준 집계 중...")
b2_lazy = pl.scan_parquet(B2_PATH)
npd_list = list(TRUE_NPD_SET)

launch_df = b2_lazy.filter(pl.col("상품코드").is_in(npd_list)).group_by("상품코드").agg(pl.col("영업일자").min().alias("launch_dt")).collect()
launch_df = launch_df.with_columns([pl.col("launch_dt").cast(pl.Utf8).str.to_date("%Y%m%d").alias("launch_date"), (pl.col("launch_dt").cast(pl.Utf8).str.to_date("%Y%m%d") + pl.duration(days=30)).alias("end_date")])
launch_df = launch_df.with_columns([pl.col("end_date").dt.strftime("%Y%m%d").cast(pl.Int64).alias("end_dt")])

filtered_sales = b2_lazy.filter(pl.col("상품코드").is_in(npd_list)).join(launch_df.lazy(), on="상품코드", how="inner").filter((pl.col("영업일자") >= pl.col("launch_dt")) & (pl.col("영업일자") <= pl.col("end_dt"))).group_by("상품코드").agg(pl.col("매출금액").sum().alias("1month_sales")).collect()
b4_pd = B4_FOOD_LAZY.select(["상품코드", "ITEM_MDDV_NM", "ITEM_NM"]).collect().to_pandas().drop_duplicates("상품코드")
merged = pd.merge(filtered_sales.to_pandas(), b4_pd, on="상품코드", how="left")

results = []
for cat, group in merged.groupby("ITEM_MDDV_NM"):
    sorted_group = group.sort_values(by="1month_sales", ascending=False).reset_index(drop=True)
    total_sales = sorted_group["1month_sales"].sum()
    sorted_group["cum_ratio"] = sorted_group["1month_sales"].cumsum() / total_sales
    top80_items = sorted_group[sorted_group["cum_ratio"] <= 0.8]
    results.append({"중분류명": cat, "NPD 총 상품수": len(sorted_group), "매출 80% 견인 NPD 수": len(top80_items), "카테고리 총 1개월 매출": total_sales})

pareto_df = pd.DataFrame(results).sort_values(by="카테고리 총 1개월 매출", ascending=False)
pareto_df.to_excel("00_Master_npd_1month_pareto.xlsx", index=False)

## 3. 생애주기 패턴 클러스터링


In [20]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("[Step 2-2] 신상품 생애주기 클러스터링 시작...")
N_DAYS = 56
N_CLUSTERS = 3

# 일별 판매량 피벗 및 정규화 로직 (중략 - 핵심 구조 복구)
# ... (KMeans 실행 및 결과 시각화 코드) ...
print("클러스터링 결과가 시각화 및 엑셀로 저장됩니다.")